<a href="https://colab.research.google.com/github/Belcamino/Ex1.Precessione/blob/master/precessione.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================================
# PRECESSIONE DEL PERIELIO DI MERCURIO
# ==========================================================================
# Questo script simula l'orbita di Mercurio usando l'equazione relativistica
# (correzione di Schwarzschild al primo ordine post-newtoniano) e mostra
# come il perielio (punto di massimo avvicinamento al Sole) ruoti nel tempo.
#
# NOTA SULLA SCALA: la precessione relativistica reale di Mercurio è di
# soli ~43 secondi d'arco per secolo (circa 0.43" per orbita) -> del tutto
# invisibile in un grafico dopo poche orbite. Per questo motivo il termine
# relativistico viene moltiplicato per un FATTORE_SCALA (variabile, si
# consiglia di partire da qualche milione) che esagera l'effetto solo a
# scopo didattico/visivo. Il valore fisico reale viene comunque calcolato
# e stampato a parte, per confronto.
#
# Da eseguire in Google Colab: basta incollare tutto in una cella ed
# eseguire (richiede solo numpy, scipy, matplotlib, già presenti in Colab).
# ==========================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# --------------------------------------------------------------------------
# 1) COSTANTI FISICHE (SI: metri, secondi, kg)
# --------------------------------------------------------------------------
G = 6.67430e-11          # Costante di gravitazione universale [m^3 kg^-1 s^-2]
C = 2.99792458e8         # Velocità della luce nel vuoto [m/s]
M_SOLE = 1.98892e30      # Massa del Sole [kg]
GM = G * M_SOLE          # Parametro gravitazionale standard del Sole [m^3/s^2]

# --------------------------------------------------------------------------
# 2) PARAMETRI ORBITALI DI MERCURIO (dati osservativi reali)
# --------------------------------------------------------------------------
A_MERCURIO = 5.7909e10   # Semiasse maggiore dell'orbita [m]
E_MERCURIO = 0.2056      # Eccentricità dell'orbita (adimensionale)

# Semi-latus rectum (parametro geometrico dell'ellisse): p = a(1-e^2)
P_ORBITA = A_MERCURIO * (1 - E_MERCURIO**2)   # [m]

# --------------------------------------------------------------------------
# 3) FATTORE DI SCALA PER LA VISUALIZZAZIONE
# --------------------------------------------------------------------------
# Moltiplica il termine relativistico (3GM/c^2 * u^2) per rendere la
# precessione visibile in poche orbite. Prova a cambiare questo valore:
# - troppo basso (es. 1)        -> l'ellisse sembra chiusa, nessuna rotazione visibile
# - valore consigliato: 1e6     -> il perielio avanza di ~29° per orbita, ben visibile
# - troppo alto (es. 1e7)       -> il perielio avanza di ~290° per orbita: le orbite
#                                  si sovrappongono quasi completamente, poco leggibile
FATTORE_SCALA = 1e6

# --------------------------------------------------------------------------
# 4) NUMERO DI ORBITE DA SIMULARE
# --------------------------------------------------------------------------
N_ORBITE = 4              # Quante rivoluzioni complete disegnare
PUNTI_PER_ORBITA = 2000   # Risoluzione angolare della simulazione

# --------------------------------------------------------------------------
# 5) EQUAZIONE DEL MOTO (in coordinate u = 1/r, variabile indipendente = phi)
# --------------------------------------------------------------------------
# Equazione classica (Newton):      d^2u/dphi^2 + u = GM / L^2
# Correzione relativistica (Schwarzschild, 1° ordine):
#                                    d^2u/dphi^2 + u = GM/L^2 + (3GM/c^2) * u^2
#
# Il termine 3GM/c^2 * u^2 è ciò che fa "precedere" l'ellisse: cresce quando
# u è grande, cioè vicino al perielio, spingendo l'orbita a ruotare proprio
# nel punto di massimo avvicinamento al Sole.
#
# Per l'orbita kepleriana classica vale: GM/L^2 = 1/p, quindi usiamo
# direttamente 1/P_ORBITA al posto di GM/L^2.

def equazione_orbita(phi, stato):
    u, du_dphi = stato
    d2u_dphi2 = (1.0 / P_ORBITA) - u + FATTORE_SCALA * (3.0 * GM / C**2) * u**2
    return [du_dphi, d2u_dphi2]

# --------------------------------------------------------------------------
# 6) CONDIZIONI INIZIALI: si parte dal perielio (massimo avvicinamento)
# --------------------------------------------------------------------------
r_perielio = A_MERCURIO * (1 - E_MERCURIO)   # distanza minima dal Sole [m]
u0 = 1.0 / r_perielio                        # u iniziale
du0 = 0.0                                    # al perielio dr/dphi = 0 (punto di inversione)

stato_iniziale = [u0, du0]
phi_span = (0, N_ORBITE * 2 * np.pi)         # intervallo angolare totale da integrare
phi_eval = np.linspace(*phi_span, N_ORBITE * PUNTI_PER_ORBITA)

# --------------------------------------------------------------------------
# 7) INTEGRAZIONE NUMERICA DELL'EQUAZIONE DIFFERENZIALE
# --------------------------------------------------------------------------
soluzione = solve_ivp(
    equazione_orbita,
    phi_span,
    stato_iniziale,
    t_eval=phi_eval,
    method='RK45',
    rtol=1e-10,
    atol=1e-12
)

phi = soluzione.t
u = soluzione.y[0]
r = 1.0 / u                       # torniamo alla distanza radiale [m]

# Coordinate cartesiane dell'orbita
x = r * np.cos(phi)
y = r * np.sin(phi)

# --------------------------------------------------------------------------
# 8) CALCOLO DELLA PRECESSIONE REALE (senza fattore di scala), per confronto
# --------------------------------------------------------------------------
# Formula analitica della precessione per orbita (radianti):
#   delta_phi = 6 * pi * GM / (c^2 * p)
precessione_reale_rad_per_orbita = 6 * np.pi * GM / (C**2 * P_ORBITA)
precessione_reale_arcsec_per_orbita = np.degrees(precessione_reale_rad_per_orbita) * 3600

# Periodo orbitale di Mercurio in anni (III legge di Keplero) e orbite/secolo
periodo_mercurio_anni = np.sqrt(A_MERCURIO**3 / GM) * 2 * np.pi / (3600*24*365.25)
orbite_per_secolo = 100 / periodo_mercurio_anni
precessione_reale_arcsec_per_secolo = precessione_reale_arcsec_per_orbita * orbite_per_secolo

print("=== Precessione reale del perielio di Mercurio (senza fattore di scala) ===")
print(f"Periodo orbitale: {periodo_mercurio_anni:.3f} anni")
print(f"Precessione per orbita: {precessione_reale_arcsec_per_orbita:.5f} arcsec")
print(f"Precessione per secolo: {precessione_reale_arcsec_per_secolo:.2f} arcsec/secolo")
print("(Il valore storicamente noto, confermato da Einstein nel 1915, è ~43 arcsec/secolo)")
print()
print(f"Fattore di scala usato nella simulazione grafica: {FATTORE_SCALA:.0e}")
print(f"Precessione 'amplificata' per orbita nel grafico: "
      f"{np.degrees(precessione_reale_rad_per_orbita * FATTORE_SCALA):.2f} gradi")

# --------------------------------------------------------------------------
# 9) VISUALIZZAZIONE ANIMATA E INTERATTIVA (Plotly)
# --------------------------------------------------------------------------
# Usiamo Plotly (preinstallato in Colab) perché offre di serie i controlli
# di animazione: un pulsante Play, un pulsante Pausa e uno slider per
# scorrere manualmente il tempo. In più aggiungiamo dei pulsanti extra per
# cambiare la VELOCITA' di riproduzione (lento / normale / veloce), agendo
# sulla durata di ciascun frame.
import plotly.graph_objects as go

# Sottocampioniamo i punti calcolati per ottenere un'animazione fluida
# senza generare migliaia di frame (che sarebbero lenti da renderizzare)
N_FRAME = 240                                  # numero di "fotogrammi" dell'animazione
indici_frame = np.linspace(0, len(x) - 1, N_FRAME).astype(int)

# Durata di un frame in millisecondi per le tre velocità disponibili:
# valori piu' bassi = animazione piu' veloce
DURATA_FRAME_LENTO_MS = 120     # velocità 0.5x
DURATA_FRAME_NORMALE_MS = 40    # velocità 1x
DURATA_FRAME_VELOCE_MS = 10     # velocità 3x

# Costruiamo un frame per ogni indice: ogni frame contiene
# 1) la "scia" percorsa fino a quel momento (linea)
# 2) la posizione attuale di Mercurio (marker)
frames = []
for k in indici_frame:
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[0], y=[0], mode='markers',
                       marker=dict(color='orange', size=22), name='Sole'),
            go.Scatter(x=x[:k+1], y=y[:k+1], mode='lines',
                       line=dict(color='royalblue', width=1.5), name='Orbita percorsa'),
            go.Scatter(x=[x[k]], y=[y[k]], mode='markers',
                       marker=dict(color='crimson', size=10), name='Mercurio')
        ],
        name=str(k)
    ))

# Figura iniziale (frame 0): Sole fisso + orbita iniziale + Mercurio in partenza
fig = go.Figure(
    data=[
        go.Scatter(x=[0], y=[0], mode='markers',
                   marker=dict(color='orange', size=22), name='Sole'),
        go.Scatter(x=x[:1], y=y[:1], mode='lines',
                   line=dict(color='royalblue', width=1.5), name='Orbita percorsa'),
        go.Scatter(x=[x[0]], y=[y[0]], mode='markers',
                   marker=dict(color='crimson', size=10), name='Mercurio')
    ],
    frames=frames
)

# NOTA: ridefiniamo il Sole in ogni frame (anche se resta sempre in (0,0))
# perché in Plotly, se il numero o l'ordine dei trace non è coerente tra un
# frame e l'altro, alcuni elementi possono non essere ridisegnati correttamente.

# Range fisso degli assi, calcolato su tutta l'orbita con un margine del 10%,
# cosi la vista non si sposta/ridimensiona durante l'animazione
margine = 0.1
estensione = max(np.abs(x).max(), np.abs(y).max())
limite_min = -estensione * (1 + margine)
limite_max = estensione * (1 + margine)

def pulsante_play(durata_ms, etichetta):
    """Crea un pulsante che avvia l'animazione con una data velocità."""
    return dict(
        label=etichetta,
        method='animate',
        args=[None, dict(frame=dict(duration=durata_ms, redraw=True),
                          fromcurrent=True, mode='immediate',
                          transition=dict(duration=0))]
    )

pulsante_pausa = dict(
    label='⏸ Pausa',
    method='animate',
    args=[[None], dict(frame=dict(duration=0, redraw=False),
                        mode='immediate', transition=dict(duration=0))]
)

fig.update_layout(
    title=f'Precessione del perielio di Mercurio (animata)<br>'
          f'<sup>fattore di scala relativistico = {FATTORE_SCALA:.0e}</sup>',
    xaxis_title='x [m]',
    yaxis_title='y [m]',
    width=750, height=750,
    # Range degli assi FISSO, calcolato sull'intera orbita con un margine del 10%.
    # Senza questo, Plotly riscala automaticamente gli assi frame per frame
    # in base ai soli punti disegnati in quel momento: il Sole sembra "sparire"
    # e l'orbita si vede "a pezzi" perché la vista si sposta e si ridimensiona
    # ad ogni fotogramma invece di restare fissa sulla scena completa.
    xaxis=dict(range=[limite_min, limite_max], scaleanchor='y', scaleratio=1,
               autorange=False, constrain='domain'),
    yaxis=dict(range=[limite_min, limite_max], autorange=False),
    uirevision='fisso',   # evita che Plotly resetti/riscali la vista durante l'interazione
    updatemenus=[dict(
        type='buttons',
        direction='left',
        x=0.0, y=1.12, xanchor='left', yanchor='top',
        buttons=[
            pulsante_play(DURATA_FRAME_LENTO_MS, '🐢 Lento (0.5x)'),
            pulsante_play(DURATA_FRAME_NORMALE_MS, '▶ Play (1x)'),
            pulsante_play(DURATA_FRAME_VELOCE_MS, '🚀 Veloce (3x)'),
            pulsante_pausa,
        ]
    )],
    sliders=[dict(                       # slider per scorrere manualmente l'animazione
        steps=[dict(method='animate',
                    args=[[str(k)], dict(mode='immediate',
                                          frame=dict(duration=0, redraw=True),
                                          transition=dict(duration=0))],
                    label='')
               for k in indici_frame],
        x=0, y=-0.05, len=1.0,
        currentvalue=dict(prefix='Fotogramma: ')
    )]
)

fig.show()

=== Precessione reale del perielio di Mercurio (senza fattore di scala) ===
Periodo orbitale: 0.241 anni
Precessione per orbita: 0.10354 arcsec
Precessione per secolo: 43.00 arcsec/secolo
(Il valore storicamente noto, confermato da Einstein nel 1915, è ~43 arcsec/secolo)

Fattore di scala usato nella simulazione grafica: 1e+06
Precessione 'amplificata' per orbita nel grafico: 28.76 gradi
